# Lab 3: Diagnosing Image Failures

**Workshop 2, block 2. 25 minutes.**

Level 3 is 40% of the preliminary round, and most teams score zero on it
after the homework. That is the expected starting point.

When the answer lives inside a JPEG, no amount of chunking, reranking or
prompt tuning reaches it. But "my bot cannot answer image questions" has
three causes needing three different fixes:

| Cause | What happened | Fix |
|---|---|---|
| **A** Never collected | The scraper ignored `img` tags | Collect them while scraping |
| **B** Never described | You have the URL but nothing turned the picture into text | Describe at ingestion |
| **C** Described badly | The description is a caption, not an extraction | Lab 4 |

Sort each failing question into a column. Most teams have all three.

> **Before you start:** `data/chroma` from lab 2, and `dev_set.json`.
>
> **When you finish:** `data/images.json`, an inventory of every image you can find. Lab 4 describes them.

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [ ]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

In [ ]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

---
## Step 1: Which Lv3 questions do you fail?

In [ ]:
store = get_store("data/chroma", name="workshop")   # built in lab 2
dev   = load_dev_set("dev_set.json")
lv3   = [d for d in dev if d["level"] == 3]

for item in lv3:
    print("=" * 70)
    print("Q:", item["question"])
    print("expected:", item["answer"])
    print("-" * 70)
    show(query(store, item["question"], k=5), chars=160)

---
## Step 2: Cause A, was the image ever collected?

If your scraper only kept text, this is your problem and nothing
downstream matters yet.

In [ ]:
img_file = Path("data/images.json")
if img_file.exists():
    images = json.loads(img_file.read_text())
    print(f"{len(images)} image records already collected")
else:
    print("No image records found. That is Cause A.")
    print("Run the next cell to build the inventory.")

### Collect them

Do this in the same pass as the text. Going back for images later means
crawling both sites a second time.

Alt text and captions are already text: index them regardless. They cost
nothing and occasionally contain the answer with no vision model
involved at all.

In [ ]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import requests

def extract_images(html, page_url):
    soup, out = BeautifulSoup(html, "html.parser"), []
    for img in soup.select("img"):
        src = img.get("src")
        if not src:
            continue
        fig = img.find_parent("figure")
        cap = (fig.find("figcaption").get_text(strip=True)
               if fig and fig.find("figcaption") else "")
        out.append({
            "src":     urljoin(page_url, src),    # relative -> absolute
            "alt":     img.get("alt", ""),
            "caption": cap,
            "page":    page_url,
        })
    return out


PAGES = [
    # TODO: your crawled page URLs
    "https://innowings.engg.hku.hk/innowing1/",
]

images = []
for url in PAGES:
    try:
        images += extract_images(requests.get(url, timeout=20).text, url)
    except Exception as exc:
        print("failed:", url, exc)

Path("data").mkdir(exist_ok=True)
Path("data/images.json").write_text(json.dumps(images, indent=1))
print(f"{len(images)} images -> data/images.json")
for im in images[:5]:
    print(" ", im["src"][:78], "|", (im["alt"] or "(no alt)")[:40])

---
## Step 3: Cause B, was it ever described?

An image URL in your data is not searchable. Something has to turn the
picture into words before it can be embedded and matched.

**Describe at ingestion, never at query time.** Ingestion is unlimited;
runtime is 30 seconds.

In [ ]:
sample = store.get(limit=500)
kinds = {}
for m in sample["metadatas"]:
    kinds[m.get("kind", "text")] = kinds.get(m.get("kind", "text"), 0) + 1
print("chunk kinds in your index:", kinds)

if "image" not in kinds:
    print("\nCause B: no image descriptions in the index. Lab 4 fixes this.")

---
## Step 4: Cause C, was it described badly?

Run one image through a naive prompt and read what comes back. Then ask
whether the Lv3 question could be answered from that text alone.

In [ ]:
# TODO fill in the names of your own image files.
# You need at least 5. Put them in an images/ folder next to this notebook.
# At least one must have legible text on a sign, poster or wall, because
# transcription is the whole point of the next lab.
IMAGES = [
    "images/",   # <-- e.g. "images/makerspace_a.jpg"
    "images/",
    "images/",
    "images/",
    "images/",
]
IMAGES = [p for p in IMAGES if Path(p).is_file()]
assert len(IMAGES) >= 5, f"Fill in at least 5 image paths. Found {len(IMAGES)}."

naive = describe_image(IMAGES[0], "Describe this image.")
print(naive)
print("\nNow pick two facts that are visible in that image but absent")
print("from the description above. Those are the facts your bot cannot")
print("reach, and lab 4 is about closing that gap.")

---
## Record your diagnosis

For each Lv3 question you fail, write A, B or C. Fix in that order.

Lab 4 is entirely about C, because once A and B are done it is the thing
that sets your ceiling.